# UnifyWeaver বংশলতিকা টিউটোরিয়াল

এই ইন্টারঅ্যাক্টিভ নোটবুকটি দেখায় কীভাবে Prolog প্রেডিকেটকে Bash স্ক্রিপ্টে কম্পাইল করতে UnifyWeaver ব্যবহার করবেন।

## পূর্বশর্ত

- SWI-Prolog ইনস্টল করা থাকতে হবে
- UnifyWeaver লাইব্রেরি উপলব্ধ থাকতে হবে
- Prolog Jupyter কার্নেল ইনস্টল থাকতে হবে (`pip install prolog-jupyter-kernel`)

## শেখার উদ্দেশ্য

এই নোটবুকের শেষে, আপনি সক্ষম হবেন:
1. Prolog তথ্য এবং নিয়ম সংজ্ঞায়িত করতে
2. প্রেডিকেটগুলোকে Bash-এ কম্পাইল করতে UnifyWeaver ব্যবহার করতে
3. জেনারেট করা Bash স্ক্রিপ্টগুলো পরীক্ষা করতে
4. ট্রানজিটিভ ক্লোজার সংকলন বুঝতে

## ধাপ ১: UnifyWeaver পরিবেশ শুরু করুন

প্রথমে আমাদের UnifyWeaver মডিউলগুলো লোড করতে হবে। আমরা education ডিরেক্টরি থেকে `init.pl` ফাইলটি ব্যবহার করব।

In [ ]:
% সূচনাকরণ ফাইল লোড করুন
['../init'].

## ধাপ ২: পারিবারিক সম্পর্ক সংজ্ঞায়িত করুন

আসুন বাইবেলের বংশলতিকা থেকে পিতা-মাতা এবং সন্তানের কিছু সম্পর্ক সংজ্ঞায়িত করি।

In [ ]:
% parent তথ্য সংজ্ঞায়িত করুন
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## ধাপ ৩: প্যারেন্ট সংক্রান্ত প্রশ্ন পরীক্ষা করুন

কম্পাইল করার আগে, আসুন কয়েকটি Prolog কোয়েরি দিয়ে যাচাই করি যে আমাদের ডেটা সঠিক আছে কিনা।

In [ ]:
% কোয়েরি: আব্রাহামের সন্তান কারা?
parent(abraham, Child).

In [ ]:
% কোয়েরি: যাকোবের সন্তান কারা?
parent(jacob, Child).

## ধাপ ৪: পূর্বপুরুষ সম্পর্ক সংজ্ঞায়িত করুন

এখন আমরা ট্রানজিটিভ ক্লোজার — `ancestor` সম্পর্ক — সংজ্ঞায়িত করি।

In [ ]:
% parent-এর ট্রানজিটিভ ক্লোজার হিসেবে ancestor সংজ্ঞায়িত করুন
:- dynamic ancestor/2.

% বেস কেস: পিতা-মাতা একজন পূর্বপুরুষ
ancestor(X, Y) :- parent(X, Y).

% রিকার্সিভ কেস: যদি X, Y-এর পিতা-মাতা হয় এবং Y, Z-এর পূর্বপুরুষ হয়, তবে X, Z-এর পূর্বপুরুষ
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## ধাপ ৫: পূর্বপুরুষ সংক্রান্ত প্রশ্ন পরীক্ষা করুন

আসুন যাচাই করি যে আমাদের পূর্বপুরুষ প্রেডিকেটটি সঠিকভাবে কাজ করছে কিনা।

In [ ]:
% কোয়েরি: আব্রাহাম কি যাকোবের পূর্বপুরুষ?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% কোয়েরি: আব্রাহামের সমস্ত বংশধর কারা?
ancestor(abraham, Descendant).

## ধাপ ৬: Parent-কে Bash-এ কম্পাইল করুন

এখন আকর্ষণীয় অংশ — আসুন আমাদের `parent/2` তথ্যগুলোকে একটি Bash স্ক্রিপ্টে কম্পাইল করি!

In [ ]:
% স্ট্রিম কম্পাইলার লোড করুন
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % parent তথ্যকে bash-এ কম্পাইল করুন
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## ধাপ ৭: Parent স্ক্রিপ্ট সংরক্ষণ করুন

আসুন জেনারেট করা Bash কোডটি একটি ফাইলে সংরক্ষণ করি।

In [ ]:
% ফাইলে সংরক্ষণ করুন
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## ধাপ ৮: Ancestor-কে Bash-এ কম্পাইল করুন

এখন `ancestor/2` প্রেডিকেটটি কম্পাইল করুন, যা রিকার্শন ব্যবহার করে।

In [ ]:
% রিকার্সিভ কম্পাইলার লোড করুন
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % ancestor-কে bash-এ কম্পাইল করুন
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## ধাপ ৯: Ancestor স্ক্রিপ্ট সংরক্ষণ করুন

পূর্বপুরুষ স্ক্রিপ্টটি একটি ফাইলে সংরক্ষণ করুন।

In [ ]:
% ফাইলে সংরক্ষণ করুন
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## ধাপ ১০: জেনারেট করা স্ক্রিপ্টগুলো পরীক্ষা করুন

এখন আমাদের তৈরি করা Bash স্ক্রিপ্টগুলো পরীক্ষা করুন! bash কমান্ড চালানোর জন্য আমরা `%%bash` ম্যাজিক ব্যবহার করব।

In [ ]:
%%bash
# parent স্ক্রিপ্টটি source করুন
source ../output/parent.sh

# পরীক্ষা: আব্রাহামের সন্তান কারা?
echo "আব্রাহামের সন্তান:"
parent abraham

In [ ]:
%%bash
# উভয় স্ক্রিপ্ট source করুন
source ../output/parent.sh
source ../output/ancestor.sh

# পরীক্ষা: আব্রাহামের বংশধর কারা?
echo "আব্রাহামের বংশধর:"
ancestor abraham

In [ ]:
%%bash
# উভয় স্ক্রিপ্ট source করুন
source ../output/parent.sh
source ../output/ancestor.sh

# পরীক্ষা: আব্রাহাম কি যিহূদার পূর্বপুরুষ?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ হ্যাঁ, আব্রাহাম যিহূদার পূর্বপুরুষ"
else
    echo "✗ না"
fi

## ধাপ ১১: সংকলন কৌশল বোঝা

আসুন বিশ্লেষণ করি UnifyWeaver কী করেছে:

1. **Parent সংকলন**: সমস্ত পিতা-মাতা-সন্তানের জোড়া তৈরি করার জন্য একটি সাধারণ স্ট্রিমিং ফাংশন তৈরি করতে `stream_compiler` ব্যবহার করেছে

2. **Ancestor সংকলন**: ট্রানজিটিভ ক্লোজার প্যাটার্ন শনাক্ত করেছে এবং দক্ষতার সাথে সমস্ত পৌঁছনোর যোগ্য পূর্বপুরুষ গণনা করতে BFS (ব্রেডথ-ফার্স্ট সার্চ) অপ্টিমাইজেশন প্রয়োগ করেছে

আসুন সংকলন কৌশলটি যাচাই করি:

In [ ]:
% ancestor রিকার্সিভ হিসেবে শ্রেণীবদ্ধ কিনা তা পরীক্ষা করুন
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## সারসংক্ষেপ

এই নোটবুকে আপনি শিখেছেন:

✅ কীভাবে Prolog তথ্য এবং নিয়ম সংজ্ঞায়িত করতে হয়

✅ তথ্যের জন্য কীভাবে UnifyWeaver-এর `stream_compiler` ব্যবহার করতে হয়

✅ রিকার্সিভ প্রেডিকেটের জন্য কীভাবে UnifyWeaver-এর `recursive_compiler` ব্যবহার করতে হয়

✅ কীভাবে তৈরি হওয়া Bash স্ক্রিপ্ট পরীক্ষা করতে হয়

✅ UnifyWeaver কীভাবে স্বয়ংক্রিয়ভাবে ট্রানজিটিভ ক্লোজার শনাক্ত করে এবং BFS অপ্টিমাইজেশন প্রয়োগ করে

## পরবর্তী পদক্ষেপ

এই অনুশীলনগুলো চেষ্টা করুন:

1. বংশলতিকায় পরিবারের আরও সদস্য যুক্ত করুন
2. একটি `grandparent/2` প্রেডিকেট সংজ্ঞায়িত করুন এবং এটি কম্পাইল করুন
3. একটি `sibling/2` প্রেডিকেট তৈরি করুন (একই পিতা-মাতার দুই সন্তান)
4. BFS অ্যালগরিদমটি বোঝার জন্য জেনারেট করা Bash কোডটি অন্বেষণ করুন

উন্নত রিকার্শন প্যাটার্ন সম্পর্কে জানতে **নোটবুক ২: রিকার্শন প্যাটার্ন তুলনা**-তে এগিয়ে যান!